# 01 Data Preprocessing

Inspect labels, validate the transaction graph and save chronological split indices. No predictive model is trained in this notebook. Report: Section 2.1 and Appendix A.


## 1. Setup
Set the data path, choose a runtime and install the dependencies used below. The `quick` profile checks the workflow with a small budget; use `report` for the report experiments. Place the four original CSV files in one folder.


In [ ]:
from pathlib import Path
# Use the same data directory and run name in all notebooks.
PROFILE = globals().get('PROFILE', 'quick')  # quick: smoke test; report: full training
DATA_SOURCE = globals().get('DATA_SOURCE', 'drive')  # drive, upload or local
DATA_DIR = Path(globals().get('DATA_DIR', '/content/drive/MyDrive/bt-4012-competition-2026'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/content/drive/MyDrive/BT4012_Final_Report_English'))
RUN_NAME = globals().get('RUN_NAME', 'final_report_english_v1')
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
EXPECTED_COMPETITION = globals().get('EXPECTED_COMPETITION', True)
assert PROFILE in ['quick', 'report']
OUT = OUTPUT_ROOT / RUN_NAME / PROFILE


In [ ]:
import sys, subprocess
# Keep the torch/CUDA version supplied by Colab.
PACKAGES = ['numpy>=1.26', 'pandas>=2.0', 'scipy>=1.11', 'matplotlib']
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)


In [ ]:
CSV_FILES = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif DATA_SOURCE == 'upload':
    from google.colab import files
    DATA_DIR = Path('/content/bt4012_data')
    DATA_DIR.mkdir(exist_ok=True)
    missing = [name for name in CSV_FILES if not (DATA_DIR / name).exists()]
    if missing:
        print('Upload the missing files:', missing)
        for name, content in files.upload().items():
            if name not in missing:
                raise ValueError('Unexpected file: ' + name)
            (DATA_DIR / name).write_bytes(content)
    if str(OUTPUT_ROOT).startswith('/content/drive/'):
        OUTPUT_ROOT = Path('/content/BT4012_Final_Report_English')
        OUT = OUTPUT_ROOT / RUN_NAME / PROFILE
elif DATA_SOURCE != 'local':
    raise ValueError(DATA_SOURCE)
for name in CSV_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUT.mkdir(parents=True, exist_ok=True)
print('Mode:', PROFILE, '| Output:', OUT)


## 2. Required functions
Only the data structures, models and helpers used by this notebook are included below.


In [ ]:
from pathlib import Path
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
from collections import deque
import hashlib
import json
import numpy as np
import pandas as pd

FEATURES = [f'feat_{i}' for i in range(1, 166)]

def serial(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial, allow_nan=False))
    temp.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serial).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b''):
            h.update(block)
    return h.hexdigest()

class Dataset:
    """Read the four CSV files and build the transaction graph."""

    def __init__(self, data_dir, expected_competition=True):
        self.data_dir = Path(data_dir)
        self.train = pd.read_csv(self.data_dir / 'train.csv')
        self.test = pd.read_csv(self.data_dir / 'test.csv')
        self.sample = pd.read_csv(self.data_dir / 'sample_submission.csv')
        raw_edges = pd.read_csv(self.data_dir / 'txs_edgelist.csv')
        self.duplicate_edges = int(raw_edges.duplicated().sum())
        self.edges = raw_edges.drop_duplicates(['txId1', 'txId2']).reset_index(drop=True)
        if list(self.train.columns) != ['txId', 'time_step', *FEATURES, 'label']:
            raise ValueError('Unexpected train columns/order')
        if list(self.test.columns) != ['index', 'txId', 'time_step', *FEATURES]:
            raise ValueError('Unexpected test columns/order')
        if list(self.sample.columns) != ['index', 'target']:
            raise ValueError('Submission schema')
        if self.train.txId.duplicated().any() or self.test.txId.duplicated().any():
            raise ValueError('Duplicate IDs')
        if set(self.train.txId) & set(self.test.txId):
            raise ValueError('Overlapping IDs')
        if not np.array_equal(self.sample['index'], self.test['index']):
            raise ValueError('Submission order')
        if not self.test['index'].is_unique:
            raise ValueError('Duplicate submission IDs')
        if not self.train.label.dropna().isin([0, 1]).all():
            raise ValueError('Labels must be 0/1/blank')
        for df in [self.train, self.test]:
            if not np.isfinite(df[FEATURES].to_numpy()).all():
                raise ValueError('Nonfinite raw features')
        if expected_competition:
            assert (len(self.train), len(self.test), len(self.edges)) == (141772, 15329, 234355)
            assert self.train.time_step.between(1, 35).all() and self.test.time_step.between(36, 49).all()
        self.ids = np.unique(np.r_[self.train.txId, self.test.txId, self.edges.to_numpy().ravel()])
        self.src = np.searchsorted(self.ids, self.edges.txId1)
        self.dst = np.searchsorted(self.ids, self.edges.txId2)
        n = len(self.ids)
        self.outgoing = coo_matrix((np.ones(len(self.src)), (self.src, self.dst)), shape=(n, n)).tocsr()
        self.incoming = self.outgoing.T.tocsr()
        self.indeg = np.diff(self.incoming.indptr)
        self.outdeg = np.diff(self.outgoing.indptr)
        self.nc, self.components = connected_components(self.outgoing, directed=False)
        all_rows = pd.concat([self.train, self.test], ignore_index=True).set_index('txId').reindex(self.ids)
        self.node_raw = all_rows[FEATURES].to_numpy(np.float32)
        self.has_features = np.isfinite(self.node_raw).all(1)
        self.node_y = all_rows.label.fillna(-1).to_numpy(np.int64)
        observed = all_rows.time_step.to_numpy(float)
        self.steps = np.full(n, -1, np.int64)
        for c in range(self.nc):
            member = self.components == c
            times = np.unique(observed[member & np.isfinite(observed)])
            if len(times) != 1:
                raise ValueError('Component does not identify exactly one snapshot')
            self.steps[member] = int(times[0])
        if np.any(self.steps[self.src] != self.steps[self.dst]):
            raise ValueError('Cross-snapshot edge')
        q = deque(np.flatnonzero(self.indeg == 0).tolist())
        deg = self.indeg.copy()
        self.order = []
        while q:
            v = q.popleft()
            self.order.append(v)
            for w in self.outgoing.indices[self.outgoing.indptr[v]:self.outgoing.indptr[v + 1]]:
                deg[w] -= 1
                if deg[w] == 0:
                    q.append(int(w))
        if len(self.order) != n:
            raise ValueError('Graph is not a DAG')
        # Unknown labels are graph context, not negative examples.
        self.frame = self.train.loc[self.train.label.notna()].copy().reset_index(drop=True)
        self.frame.label = self.frame.label.astype(np.int64)
        self.pos = np.searchsorted(self.ids, self.frame.txId)
        self.testpos = np.searchsorted(self.ids, self.test.txId)
        self.node_x = np.column_stack([self.node_raw, self.steps, self.indeg, self.outdeg]).astype(np.float32)
        self.raw = self.node_x[self.pos]
        self.y = self.frame.label.to_numpy(np.int64)
        self._extras = {}
        self.data_hashes = {name: file_hash(self.data_dir / name) for name in ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']}
        self.identity = fingerprint(self.data_hashes)

    def split(self, cut, horizon=14):
        t = self.frame.time_step.to_numpy()
        a = np.flatnonzero(t <= cut)
        b = np.flatnonzero((t > cut) & (t <= cut + horizon))
        if not len(a) or not len(b):
            raise ValueError('Empty temporal split')
        return (a, b)

    def audit(self):
        a = np.isin(self.ids, self.train.txId)
        b = np.isin(self.ids, self.test.txId)
        mixed = sum((bool(a[self.components == c].any() and b[self.components == c].any()) for c in range(self.nc)))
        return dict(train_rows=len(self.train), test_rows=len(self.test), supervised_rows=len(self.frame), labels=self.train.label.value_counts().to_dict(), unknown=int(self.train.label.isna().sum()), features=165, graph_nodes=len(self.ids), graph_edges=len(self.edges), duplicate_edges=self.duplicate_edges, graph_only=int((~self.has_features).sum()), weak_components=self.nc, dag=True, mixed_train_test_components=mixed, train_steps=sorted(self.train.time_step.unique().tolist()), test_steps=sorted(self.test.time_step.unique().tolist()), data_hashes=self.data_hashes)


## 3. Load and check the data


In [ ]:
# Labelled rows are the only rows used by supervised models.
data = Dataset(DATA_DIR, expected_competition=EXPECTED_COMPETITION)
print('Labelled training rows:', len(data.frame), '| Unknown labels:', int(data.train.label.isna().sum()), '| Test rows:', len(data.test))


## 4. Labels and transaction graph
Blank labels mean unknown, not licit. Keep these transactions as graph context. Degree counts connected transactions, not raw Bitcoin inputs or outputs.


In [ ]:
audit = data.audit()
save_json(OUT / 'data_audit.json', audit)
display(pd.DataFrame({'subset': ['licit', 'illicit', 'unknown', 'test'],
    'rows': [int((data.y == 0).sum()), int((data.y == 1).sum()), int(data.train.label.isna().sum()), len(data.test)]}))
print({key: audit[key] for key in ['graph_nodes','graph_edges','graph_only','weak_components','dag','mixed_train_test_components']})
assert audit['mixed_train_test_components'] == 0


## 5. Save raw inputs and temporal splits
Save untransformed inputs. Imputation, scaling and embedding bins are fitted later, inside each model training fold.


In [ ]:
np.savez_compressed(OUT / 'raw_model_inputs.npz', X=data.raw, y=data.y,
    txId=data.frame.txId.to_numpy(), time_step=data.frame.time_step.to_numpy(),
    test_X=data.node_x[data.testpos], test_index=data.test['index'].to_numpy())
splits = {}
for cutoff in [7, 10, 14, 17, 21, 28]:
    horizon = 7 if cutoff == 28 else 14
    train_rows, valid_rows = data.split(cutoff, horizon)
    inner_train, inner_valid = data.split(cutoff - 3, 3)
    splits.update({f'cut{cutoff}_train': train_rows, f'cut{cutoff}_validation': valid_rows,
        f'cut{cutoff}_inner_train': inner_train, f'cut{cutoff}_inner_validation': inner_valid})
np.savez_compressed(OUT / 'temporal_splits.npz', **splits)
print('Saved raw inputs and split indices to:', OUT)


## 6. Label distribution and feature shift


In [ ]:
import matplotlib.pyplot as plt
summary = data.train.assign(known=data.train.label.notna()).groupby('time_step').agg(
    rows=('txId','size'), known=('known','sum'), illicit=('label','sum'))
summary['illicit_fraction'] = summary.illicit / summary.known
summary.to_csv(OUT / 'labels_by_period.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary[['rows','known']].plot(ax=axes[0], title='Transactions by period')
summary.illicit_fraction.plot(ax=axes[1], title='Illicit fraction among known labels')
plt.tight_layout(); plt.show()
fig.savefig(OUT / 'labels_by_period.png', dpi=150)
shift = []
for feature in ['feat_100','feat_101','feat_103']:
    shift.append(dict(feature=feature, training_max=float(data.frame[feature].max()),
        test_above_training_max=float((data.test[feature] > data.frame[feature].max()).mean())))
display(pd.DataFrame(shift))
save_json(OUT / 'feature_shift.json', shift)
